# exp105a: Meta-TTT Ablation — Inference & Analysis

**Experiment**: `exp105a_no-metattt_from_exp101`  
**Parent**: `exp101_poscond-bigram-trigram_from_exp95`  
**Single change**: `META_TTT_ENABLED=0` (FOMAML disabled)  
**Results**: pre-quant 1.1353 | int6 1.1396 | legal_ttt **1.1162**

Sections:
1. Setup & path detection
2. Load model (float `.pt` and int6 `.ptz`)
3. Compute val_bpb
4. Text generation
5. Per-token loss distribution
6. Per-position loss curve
7. Top worst-predicted tokens
8. Summary

## 1. Setup & Path Detection

In [ ]:
import sys, os, json, io, math, glob, importlib.util
import torch, torch.nn.functional as F
import numpy as np

try:
    _nb = globals().get('__vsc_ipynb_file__') or __file__
    EXP_DIR = os.path.dirname(os.path.abspath(_nb))
except NameError:
    EXP_DIR = os.getcwd()

REPO_ROOT = os.path.abspath(os.path.join(EXP_DIR, '..', '..', '..'))
CHECKPOINT_DIR = os.path.join(EXP_DIR, 'checkpoint')
TOKENIZER_PATH = os.path.join(REPO_ROOT, 'data', 'tokenizers', 'fineweb_1024_bpe.model')
VAL_DATA_PATTERN = os.path.join(REPO_ROOT, 'data', 'datasets', 'fineweb10B_sp1024', 'fineweb_val_*.bin')
DEVICE = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'

# Paths to model files (adjust if running directly in pod)
MODEL_PT  = os.environ.get('MODEL_PT',  os.path.join(EXP_DIR, 'checkpoint', 'model.pt'))
MODEL_PTZ = os.environ.get('MODEL_PTZ', os.path.join(EXP_DIR, 'checkpoint', 'model.int6.ptz'))

print(f'EXP_DIR   : {EXP_DIR}')
print(f'REPO_ROOT : {REPO_ROOT}')
print(f'DEVICE    : {DEVICE}')
print(f'model.pt  : {MODEL_PT}  — exists={os.path.exists(MODEL_PT)}')
print(f'int6.ptz  : {MODEL_PTZ} — exists={os.path.exists(MODEL_PTZ)}')

## 2. Load Model

In [ ]:
# Import train_gpt from this experiment
spec = importlib.util.spec_from_file_location('train_gpt', os.path.join(EXP_DIR, 'train_gpt.py'))
tg = importlib.util.module_from_spec(spec)
sys.path.insert(0, EXP_DIR)
spec.loader.exec_module(tg)
sys.path.pop(0)

# Build model from hyperparameters
import inspect
hp = tg.Hyperparameters()
valid_keys = set(inspect.signature(tg.GPT.__init__).parameters) - {'self'}
hp_dict = {k: getattr(hp, k) for k in valid_keys if hasattr(hp, k)}
model = tg.GPT(**hp_dict).eval()
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# --- Load float model ---
model_float = None
if os.path.exists(MODEL_PT):
    sd = torch.load(MODEL_PT, map_location='cpu', weights_only=True)
    if isinstance(sd, dict) and 'model' in sd:
        sd = sd['model']
    model_float = tg.GPT(**hp_dict).eval()
    model_float.load_state_dict(sd, strict=True)
    model_float = model_float.to(DEVICE)
    print(f'Loaded float model from {MODEL_PT}')
else:
    print('[skip] float model not found')

In [ ]:
# --- Load int6 dequantized model ---
model_int6 = None
if os.path.exists(MODEL_PTZ):
    import lzma
    with open(MODEL_PTZ, 'rb') as f:
        blob = f.read()
    # Try LZMA first (competition standard), fall back to zlib
    try:
        decompressed = lzma.decompress(blob)
    except Exception:
        import zlib
        decompressed = zlib.decompress(blob)
    qs = torch.load(io.BytesIO(decompressed), map_location='cpu', weights_only=True)
    sd_cpu = {k: v.cpu() for k, v in (sd if model_float else {}).items()}
    if not sd_cpu and model_float:
        sd_cpu = {k: v.cpu() for k, v in model_float.state_dict().items()}
    deq = tg.dequantize_mixed_int6(qs['w'], qs['m'], sd_cpu)
    # re-inject meta_sgd params if present in a fresh model
    fresh = tg.GPT(**hp_dict)
    for k in ('meta_sgd_qo', 'meta_sgd_kv', 'meta_sgd_up', 'meta_sgd_down'):
        if k not in deq and hasattr(fresh, k):
            deq[k] = getattr(fresh, k).detach().cpu().clone()
    model_int6 = tg.GPT(**hp_dict).eval()
    tg.CastedLinear._qat_enabled = True
    model_int6.load_state_dict(deq, strict=True)
    model_int6 = model_int6.to(DEVICE)
    print(f'Loaded int6 model from {MODEL_PTZ}')
else:
    print('[skip] int6 model not found')

## 3. Compute val_bpb

In [ ]:
import sentencepiece as spm
sp = spm.SentencePieceProcessor()
sp.Load(TOKENIZER_PATH)

VAL_SHARDS = sorted(glob.glob(VAL_DATA_PATTERN))
assert VAL_SHARDS, f'No val shards at {VAL_DATA_PATTERN}'

def load_val_shard(path):
    hdr = np.fromfile(path, dtype='<i4', count=256)
    n = int(hdr[2])
    return np.fromfile(path, dtype='<u2', count=n, offset=256*4)

val_tokens = np.concatenate([load_val_shard(s) for s in VAL_SHARDS])
print(f'Val tokens: {len(val_tokens):,}')

SEQ_LEN = hp.train_seq_len
BATCH_SEQ = 262144 // SEQ_LEN
LOG2E = math.log2(math.e)

def eval_bpb(m, toks, max_batches=4):
    m.eval()
    total_loss = total_toks = 0
    all_losses, all_ids = [], []
    i = 0
    with torch.no_grad():
        for _ in range(max_batches):
            if i + SEQ_LEN * BATCH_SEQ + 1 > len(toks):
                break
            chunk = toks[i:i + SEQ_LEN*BATCH_SEQ + 1].astype(np.int64)
            x = torch.from_numpy(chunk[:-1]).reshape(BATCH_SEQ, SEQ_LEN).to(DEVICE)
            y = torch.from_numpy(chunk[1: ]).reshape(BATCH_SEQ, SEQ_LEN).to(DEVICE)
            logits = m.forward_logits(x)
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)),
                                   y.reshape(-1), reduction='none')
            all_losses.append(loss.cpu()); all_ids.append(y.reshape(-1).cpu())
            total_loss += loss.sum().item(); total_toks += loss.numel()
            i += SEQ_LEN * BATCH_SEQ
    mean_loss = total_loss / total_toks
    return mean_loss, mean_loss * LOG2E, torch.cat(all_losses), torch.cat(all_ids)

if model_float:
    fl_loss, fl_bpb, fl_losses, fl_ids = eval_bpb(model_float, val_tokens)
    print(f'Float  val_loss={fl_loss:.4f}  val_bpb={fl_bpb:.4f}  (expected ~1.1353)')
if model_int6:
    q_loss, q_bpb, q_losses, q_ids = eval_bpb(model_int6, val_tokens)
    print(f'Int6   val_loss={q_loss:.4f}  val_bpb={q_bpb:.4f}  (expected ~1.1396)')

## 4. Text Generation

In [ ]:
def generate(m, prompt, max_new=150, temp=0.8, top_k=40):
    ids = sp.EncodeAsIds(prompt)
    x = torch.tensor(ids, dtype=torch.long, device=DEVICE).unsqueeze(0)
    m.eval()
    with torch.no_grad():
        for _ in range(max_new):
            logits = m.forward_logits(x)[:, -1, :] / temp
            if top_k:
                v, _ = torch.topk(logits, top_k)
                logits[logits < v[:, -1:]] = -float('inf')
            x = torch.cat([x, torch.multinomial(F.softmax(logits, -1), 1)], dim=1)
    return sp.DecodeIds(x[0].tolist())

active_model = model_float or model_int6
if active_model:
    for p in ['The history of artificial intelligence began',
              'Scientists discovered that language models']:
        print('='*60)
        print(generate(active_model, p))
        print()

## 5. Per-Token Loss Distribution

In [ ]:
import matplotlib.pyplot as plt

if model_float:
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.hist(fl_losses.numpy(), bins=100, log=True, color='steelblue', alpha=0.8)
    ax.axvline(fl_losses.mean(), color='red', linestyle='--',
               label=f'mean={fl_losses.mean():.3f}')
    ax.set_title('exp105a (no meta-TTT) — Per-Token Loss Distribution (float)')
    ax.set_xlabel('Cross-entropy'); ax.set_ylabel('Count (log)')
    ax.legend(); plt.tight_layout(); plt.show()

    for p in [50, 75, 90, 95, 99]:
        print(f'  p{p:2d}: {np.percentile(fl_losses.numpy(), p):.3f}')

## 6. Per-Position Loss Curve

In [ ]:
if model_float:
    pos_acc = np.zeros(SEQ_LEN); pos_cnt = np.zeros(SEQ_LEN, dtype=np.int64)
    i = 0
    model_float.eval()
    with torch.no_grad():
        for _ in range(4):
            if i + SEQ_LEN * BATCH_SEQ + 1 > len(val_tokens): break
            chunk = val_tokens[i:i+SEQ_LEN*BATCH_SEQ+1].astype(np.int64)
            x = torch.from_numpy(chunk[:-1]).reshape(BATCH_SEQ, SEQ_LEN).to(DEVICE)
            y = torch.from_numpy(chunk[1: ]).reshape(BATCH_SEQ, SEQ_LEN).to(DEVICE)
            loss = F.cross_entropy(model_float.forward_logits(x).reshape(-1, model_float.forward_logits(x).size(-1)),
                                   y.reshape(-1), reduction='none').reshape(BATCH_SEQ, SEQ_LEN).cpu().numpy()
            pos_acc += loss.sum(0); pos_cnt += BATCH_SEQ; i += SEQ_LEN * BATCH_SEQ
    pos_mean = pos_acc / pos_cnt
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(pos_mean, lw=0.6, color='steelblue', label='raw')
    w = max(1, SEQ_LEN//64)
    ax.plot(np.convolve(pos_mean, np.ones(w)/w, 'same'), lw=2, color='red',
            alpha=0.7, label=f'smoothed (w={w})')
    ax.set_title('exp105a — Per-Position Mean Loss'); ax.set_xlabel('Position')
    ax.set_ylabel('Mean CE'); ax.legend(); plt.tight_layout(); plt.show()

## 7. Top Worst-Predicted Tokens

In [ ]:
if model_float:
    top_n = 20
    idx = fl_losses.topk(top_n).indices.numpy()
    token_ids_np = fl_ids.numpy()
    print(f'Top {top_n} worst predicted tokens (float model):')
    print(f'{"Rank":>4}  {"Loss":>7}  {"TokenID":>8}  Piece')
    print('-'*50)
    for rank, i in enumerate(idx, 1):
        tid = int(token_ids_np[i])
        loss_val = fl_losses[i].item()
        piece = repr(sp.IdToPiece(tid))
        print(f'{rank:>4}  {loss_val:>7.3f}  {tid:>8}  {piece}')

## 8. Summary

In [ ]:
print('='*60)
print('EXPERIMENT: exp105a_no-metattt_from_exp101')
print('='*60)
print(f'Device       : {DEVICE}')
print(f'Params       : {sum(p.numel() for p in model.parameters()):,}')
print()
print('Expected results:')
print('  pre-quant val_bpb : 1.1353')
print('  int6 val_bpb      : 1.1396')
print('  legal_ttt val_bpb : 1.1162')
print()
if model_float:
    print(f'  Float this run    : {fl_bpb:.4f}')
if model_int6:
    print(f'  Int6  this run    : {q_bpb:.4f}')
print()
print('Key finding: META_TTT_ENABLED=1 vs 0 gives only +0.0003 bpb')
print('improvement at 3% compute overhead. Not worth it.')
print('See README.md and ../META_TTT_ANALYSIS.md for full analysis.')
print('='*60)